<div style="background-color:#1F3864; padding:25px; border-radius:8px;">
<h1 style="color:white; text-align:center; margin:0;">🌍 Global Trade Disruption & Commodity Price Prediction</h1>
<p style="color:#D9E2F3; text-align:center; margin-top:10px; font-size:15px;">
Predicting commodity price movements from real-world supply chain disruption signals -
built entirely on live API data (FRED, GDELT, UN Comtrade, World Bank)
</p>
</div>

In [33]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time

## Notebook Overview

This notebook builds a dataset and model to predict commodity price movements —
starting with WTI crude oil — using real-world supply chain disruption signals
sourced directly from public APIs, rather than a synthetic dataset.

**Data sources:**
- **FRED API** — commodity prices (oil WTI, oil Brent, natural gas, wheat, corn, gold, aluminum, iron ore)
- **GDELT** — geopolitical event intensity and global news coverage volume
- **UN Comtrade** — bilateral trade volume between countries
- **World Bank** — country-level logistics performance and economic context

**Target variable:**
Percentage change in WTI crude oil price over a forward window (e.g. 30 days),
predicted from current disruption signals. Predicting the *change* rather than
the raw price level avoids the model simply learning long-term inflation trends,
and instead focuses on how disruption events actually move prices — the same
approach used in the crash prediction project's 63-day-forward target.

**Major real-world events this data is expected to capture:**
COVID-19 Supply Chain Shock (2020), Russia-Ukraine Conflict (2022),
Red Sea Shipping Crisis (2024), Strait of Hormuz Disruption (2026).

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">1. Create the Data</h2>
</div>

<div style="background-color:#EDEAE5; padding:10px 15px; border-radius:6px;">
<h3 style="color:#1F3864; text-align:center; margin:0;">1.1 Commodities</h3>
</div>

In [2]:
api_key = "a3b1c076b07a348c2f9a24b0799e53e2"

In [23]:
start_date = "2000-01-01"

In [27]:
commodities = {
    "oil_wti": "DCOILWTICO",
    "oil_brent": "DCOILBRENTEU",
    "natural_gas": "DHHNGSP",
    "wheat": "PWHEAMTUSDM",
    "corn": "PMAIZMTUSDM",
    "gold": "IQ12260",
    "aluminum": "PALUMUSDM",
    "iron_ore": "PIORECRUSDM"}

In [28]:
def get_commodity_prices(commodities:dict,start_date:str)-> None :
    for name,series_id in commodities.items():
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&observation_start={start_date}&api_key={api_key}&file_type=json"
        response = requests.get(url)

        try:
            data = response.json()
        except ValueError:
            print(f"FAILED: {name} ({series_id}) — empty or invalid response, status {response.status_code}")
            continue

        if "observations" not in data:
            print(f"FAILED: {name} ({series_id}) — {data.get('error_message', 'unknown error')}")
            continue

        df = pd.DataFrame(data["observations"])
        df = df[["date", "value"]]
        df.to_csv(f"data/{name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/{name}.csv")

        time.sleep(0.5)

In [29]:
get_commodity_prices(commodities,start_date)

Saved 6931 rows to data/oil_wti.csv
Saved 6931 rows to data/oil_brent.csv
Saved 6931 rows to data/natural_gas.csv
Saved 318 rows to data/wheat.csv
Saved 318 rows to data/corn.csv
Saved 318 rows to data/gold.csv
Saved 318 rows to data/aluminum.csv
Saved 318 rows to data/iron_ore.csv


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.2 GDELT Events </h3>
</div>

In [42]:
def get_gdelt_events(queries:list)->None:
    headers = {"User-Agent": "Mozilla/5.0"}
    events = ["Strait of Hormuz", "Red Sea shipping", "Russia Ukraine conflict", "COVID supply chain"]

    for query in events:
        url = "https://api.gdeltproject.org/api/v2/doc/doc"
        params = {
            "query":f"{query}",
            "mode": "timelinevol",
            "format":"json"}
        
        response = requests.get(url , params = params , headers = headers)
        try:
            data = response.json()
        except ValueError:
            continue

        if "timeline" not in data:
            continue

        records = data["timeline"][0]["data"]
        df = pd.DataFrame(records)
        filename = query.lower().replace(" ", "_")
        df.to_csv(f"data/gdelt_{filename}.csv", index=False)
        print(f"Saved data/gdelt_{filename}.csv")
        
events = ["Strait of Hormuz", "Red Sea shipping", "Russia Ukraine conflict", "COVID supply chain"]
get_gdelt_events(events)

Saved data/gdelt_covid_supply_chain.csv


In [ ]:
get_commodity_prices(commodities,start_date)


In [45]:
primary_key = "fb672033c45d4d71a329a740d788c758"

In [46]:
url = "https://comtradeapi.un.org/data/v1/get/C/A/HS"